# 06 — Detector calibration

**Replication of:** *Presence-absence estimation in audio recordings of tropical frog communities*
(Estrella Terneux, Nicolalde, Nicolalde & Merino-Viteri, arXiv:1901.02495), Sections **3.5 to 3.8**

## 1. Objective

Notebook 05 chose the model: 20 PLP-RASTA with 4 Gaussians per species. This notebook turns it into the **detector** of the article: one frozen mixture per species, a likelihood-ratio
score for verification, and one threshold per species.

`[PAPER]` A segment is identified with the species model of highest mean frame log-likelihood (Section 3.6, Equation 9). The hypothesis is then verified with a log-likelihood ratio against the
**median of the alternative models** (Section 3.7, Equations 10 and 11), and accepted when the ratio is above a threshold chosen per species with a one-vs-all ROC curve (Section 3.8).
The article started from the operating point of a **5 % false-positive rate** and then **fine-tuned the thresholds against its manually annotated field samples**, ending with the vector `[5, 3, 6, 9, 6.75, 5.25, 5.5, 11, 6, 6]`.

### What cannot be reproduced

The fine-tuning step needs annotated pond samples, and those were not delivered. Thresholds here come from the ROC stage alone, calibrated on labelled calls. The article's own thresholds belong to its m-FCC scores and
are shown only for provenance.

### The protocol

A fixed **recording-level** split of the labelled corpus into TRAIN and CALIBRATION, so that a recording is on one side for every species. The mixtures are fitted on TRAIN, frozen, and **the same objects** are used to compute the
calibration scores and are saved for inference. Nothing is retrained after the thresholds are chosen, which is what keeps the scores seen in calibration and in the field comparable.

## 2. Setup

**What this section asks, and why it matters.** The notebook reads the selection written by notebook 05 and the per-call features of notebook 04, and writes the detector to `models/`.

In [1]:
# Every dependency, the display options, and the locations of the inputs and of the detector that is written.
import json
import os
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from joblib import Parallel, delayed

from frog_presence import detector as D
from frog_presence.gmm import CallCorpus, fit_gmm, score_calls, species_number

sns.set_theme(style="ticks")
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 140)
pd.set_option("display.max_columns", 80)

CWD = Path.cwd().resolve()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
CALLS_PATH = PROCESSED_DIR / "calls.csv"
CACHE_PATH = PROCESSED_DIR / "feature_comparison" / "call_feature_sets.joblib"
SELECTION_PATH = PROCESSED_DIR / "robust_model_selection" / "selected_robust_configuration.json"
RESULTS_DIR = PROCESSED_DIR / "final_detector_calibration"
MODELS_DIR = REPO_ROOT / "models" / "gmm_robust_final"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for path in (CALLS_PATH, CACHE_PATH, SELECTION_PATH):
    if not path.exists():
        raise FileNotFoundError(f"Missing {path.name}. Run notebooks 02, 04 and 05 first.")

# Only locations relative to the repository: the absolute paths are personal to whoever runs this.
print("Selection :", SELECTION_PATH.relative_to(REPO_ROOT).as_posix())
print("Results   :", RESULTS_DIR.relative_to(REPO_ROOT).as_posix())
print("Detector  :", MODELS_DIR.relative_to(REPO_ROOT).as_posix())

Selection : data/processed/robust_model_selection/selected_robust_configuration.json
Results   : data/processed/final_detector_calibration
Detector  : models/gmm_robust_final


## 3. Load the selection and the call features

**What this section asks, and why it matters.** The detector must be built from what notebook 05 selected, not from values remembered here. The selection names the feature and how the RASTA filter starts,
and the stored per-call features are those of the zero start, so the two have to agree before anything is fitted.

In [2]:
# Read the selected configuration and check it matches the stored features, then load the usable calls.
selection = json.loads(SELECTION_PATH.read_text(encoding="utf-8"))
FEATURE_NAME = selection["feature"]
N_COMPONENTS = int(selection["recommended_parsimonious"]["n_components"])
USABLE_CODES = tuple(selection["classes"])
MAX_FPR = 0.05

assert FEATURE_NAME == "20_PLP_RASTA" and selection["rasta_start"] == "zero", \
    "the stored per-call features are the zero-start PLP-RASTA; re-run notebook 04 for another feature"

print("feature             :", FEATURE_NAME, "| RASTA start:", selection["rasta_start"])
print("Gaussians per class :", N_COMPONENTS)
print("classes             :", USABLE_CODES)
print("excluded            :", selection["excluded_class"])

calls = pd.read_csv(CALLS_PATH).reset_index(drop=True)
calls["call_id"] = [f"call_{i:04d}" for i in range(len(calls))]
CALL_FEATURES = joblib.load(CACHE_PATH)
assert set(CALL_FEATURES) == set(calls.call_id)

corpus = CallCorpus(np.vstack([CALL_FEATURES[cid][FEATURE_NAME] for cid in calls.call_id]),
                    calls[["code", "species", "wav", "seconds", "frames"]])
usable = corpus.calls[corpus.calls.code.isin(USABLE_CODES)].copy()
usable["recording"] = usable.wav.astype(str).str.strip().str.lower()
print("\ncalls in the corpus:", len(calls), "| usable calls:", len(usable), "| recordings:", usable.recording.nunique())

feature             : 20_PLP_RASTA | RASTA start: zero
Gaussians per class : 4
classes             : ('f01', 'f02', 'f03', 'f04', 'f05', 'f06', 'f07', 'f08', 'f10')
excluded            : f09



calls in the corpus: 1461 | usable calls: 1459 | recordings: 66


`[OBSERVED]` The selection names **20 PLP-RASTA with the zero RASTA start and 4 Gaussians per class**, for the nine usable species, with `f09` excluded, and the stored per-call features are the zero-start ones, so the two agree.
Of the **1461 calls**, **1459** belong to usable species, from **66 recordings**.

## 4. A fixed recording-level TRAIN / CALIBRATION split

**What this section asks, and why it matters.** Thresholds chosen on calls the mixtures were trained on would be tuned to scores that the field will never produce. The calibration calls must come from
recordings the mixtures have not heard. Because a recording is indivisible and some species have few of them, an exact 75/25 division is impossible.

*Decisions, not stated in the article:*

- the split is **global** (a recording is on one side for every species) and **recording-level**;
- about **25 %** of each species' duration and of its calls should be in calibration, and of 10000 random candidate sets the valid one closest to that is kept (the first among equals);
- every species must be present on both sides.

In [3]:
# Search the split, then audit it: no recording or call on both sides, and every usable call on exactly one side.
CALIBRATION_FRACTION = 0.25
SPLIT_SEED = 20260821
SPLIT_ATTEMPTS = 10000
MODEL_SEED_BASE = 20260822

split = D.calibration_split(corpus.calls, USABLE_CODES, CALIBRATION_FRACTION, SPLIT_SEED, SPLIT_ATTEMPTS)
TRAIN_RECORDINGS, CALIBRATION_RECORDINGS = split["train"], split["calibration"]
assert not TRAIN_RECORDINGS & CALIBRATION_RECORDINGS

usable["side"] = np.where(usable.recording.isin(CALIBRATION_RECORDINGS), "calibration", "train")
train_calls, calibration_calls = usable[usable.side == "train"], usable[usable.side == "calibration"]
assert not set(train_calls.call_id) & set(calibration_calls.call_id)
assert len(train_calls) + len(calibration_calls) == len(usable)

print("selected attempt        :", split["attempt"], "of", SPLIT_ATTEMPTS)
print("TRAIN / CALIBRATION recordings:", len(TRAIN_RECORDINGS), "/", len(CALIBRATION_RECORDINGS),
      "| calibration share:", round(len(CALIBRATION_RECORDINGS) / usable.recording.nunique(), 4))
print("TRAIN / CALIBRATION calls     :", len(train_calls), "/", len(calibration_calls))
print("recording on both sides: False | call on both sides: False | every usable call on one side: True")

split_audit = (usable.groupby(["code", "side"]).agg(calls=("call_id", "size"), seconds=("seconds", "sum"), recordings=("recording", "nunique"))
               .unstack("side").round(2))
split_audit[("calibration_share", "seconds")] = (split_audit[("seconds", "calibration")] / split_audit["seconds"].sum(axis=1)).round(3)
display(split_audit)

selected attempt        : 1659 of 10000
TRAIN / CALIBRATION recordings: 48 / 18 | calibration share: 0.2727
TRAIN / CALIBRATION calls     : 1077 / 382
recording on both sides: False | call on both sides: False | every usable call on one side: True


calls           seconds          recordings       calibration_share
side calibration train calibration   train calibration train           seconds
code                                                                          
f01           19    79        8.64   36.08           1     4             0.193
f02           24    75        8.84   32.88           2     8             0.212
f03           26   143       15.50   87.08           1     7             0.151
f04           47   139       19.80   58.14           3     6             0.254
f05          135   205       47.56   66.80           1     2             0.416
f06           13    41        6.00   18.94           2     3             0.241
f07           17    84        9.34   48.50           1     6             0.161
f08           25    71       30.94   71.72           2     5             0.301
f10           76   240       32.18  101.82           5     7             0.240

`[OBSERVED]` The search selects attempt **1659** of 10000. The split has **48 TRAIN and 18 CALIBRATION recordings** (a calibration share of 0.2727) and **1077 and 382 calls**. No recording and no call is on both sides, and every usable call is on exactly one.

`[OBSERVED]` The calibration side holds very few recordings for several species: **1** for `f01`, `f03`, `f05` and `f07`, **2** for `f02`, `f06` and `f08`, 3 for `f04` and 5 for `f10`. The calibration share of each species' duration ranges from 0.151 (`f03`) to
0.416 (`f05`), because whole recordings cannot be divided.

## 5. Fit and freeze one mixture per species

**What this section asks, and why it matters.** These are the models that will be saved and used in the field, so they are fitted once, on **all** the labelled calls of the TRAIN recordings. The number of Gaussians is the one selected
in notebook 05; the *training duration* of that experiment was a minimum used for comparison, and discarding labelled data to match it would only weaken the detector.

In [4]:
# Fit each species' mixture on its TRAIN calls with the selected number of Gaussians, and report convergence.
FINAL_MODELS, model_rows = {}, []
for code_ in USABLE_CODES:
    mine = train_calls[train_calls.code == code_]
    model = fit_gmm(corpus.frames(tuple(mine.call_id)), N_COMPONENTS, MODEL_SEED_BASE + species_number(code_))
    FINAL_MODELS[code_] = model
    model_rows.append({"code": code_, "species": mine.species.iloc[0], "n_train_calls": len(mine),
                       "n_train_recordings": mine.recording.nunique(), "train_seconds": mine.seconds.sum(),
                       "n_frames": int(mine.frames.sum()), "converged": bool(model.converged_), "n_iter": int(model.n_iter_)})
display(pd.DataFrame(model_rows).round(2))
assert all(r["converged"] for r in model_rows)

,code,species,n_train_calls,n_train_recordings,train_seconds,n_frames,converged,n_iter
0,f01,Boana alfaroi,79,4,36.08,6979,True,31
1,f02,Dendropsophus bifurcus,75,8,32.88,6351,True,17
2,f03,Boana cinerascens,143,7,87.08,16987,True,16
3,f04,Pristimantis conspicillatus,139,6,58.14,11211,True,13
4,f05,Leptodactylus discodactylus,205,2,66.80,12745,True,29
5,f06,Osteocephalus fuscifacies,41,3,18.94,3665,True,18
6,f07,Boana lanciformis,84,6,48.50,9448,True,16
7,f08,Rhinella margaritifera,71,5,71.72,14131,True,14
8,f10,Engystomops petersi,240,7,101.82,19644,True,27


`[OBSERVED]` All nine mixtures converged, in 13 to 31 EM iterations. `f05` is fitted on 66.80 seconds of calls from only **2 training recordings**, and `f06` on 18.94 seconds from 3.

## 6. Score the calibration calls

**What this section asks, and why it matters.** Every calibration call is scored under every frozen mixture. `[PAPER]` The score is the mean frame log-likelihood (Equation 9), the hypothesis is the model with the highest score, and
the verification ratio is `LLR_k = s_k - median_{j != k} s_j`.

*Decision, not stated in the article:* the median of the alternatives is taken over the **mean log-likelihoods**, not over the likelihoods. With eight alternatives it is the mean of the middle two, and the two orders
of operations give different numbers.

In [5]:
# Score the calibration calls under all frozen models, and report the identification accuracy and the confusion matrix.
cal = calibration_calls.reset_index(drop=True)
scores = score_calls(FINAL_MODELS, corpus, list(cal.call_id))
llr = D.llr_matrix(scores)
hypothesis = scores.argmax(axis=1)
true_index = np.array([USABLE_CODES.index(c) for c in cal.code])

accuracy = float((hypothesis == true_index).mean())
print("calibration calls scored      :", len(cal))
print("raw identification accuracy   :", f"{accuracy * 100:.3f} %")

confusion = pd.DataFrame(0, index=[f"true_{c}" for c in USABLE_CODES], columns=[f"pred_{c}" for c in USABLE_CODES])
for t_, h_ in zip(true_index, hypothesis):
    confusion.iloc[t_, h_] += 1
display(confusion)

calibration calls scored      : 382
raw identification accuracy   : 97.644 %


,pred_f01,pred_f02,pred_f03,pred_f04,pred_f05,pred_f06,pred_f07,pred_f08,pred_f10
true_f01,19,0,0,0,0,0,0,0,0
true_f02,0,22,0,0,2,0,0,0,0
true_f03,0,0,26,0,0,0,0,0,0
true_f04,0,1,0,46,0,0,0,0,0
true_f05,0,3,0,0,129,0,0,0,3
true_f06,0,0,0,0,0,13,0,0,0
true_f07,0,0,0,0,0,0,17,0,0
true_f08,0,0,0,0,0,0,0,25,0
true_f10,0,0,0,0,0,0,0,0,76


`[OBSERVED]` Of the **382 calibration calls**, **97.644 %** are identified correctly, which is 9 errors: `f05` is taken for `f02` three times and for `f10` three times, `f02` for `f05` twice, and `f04` for `f02` once. The other six species are never confused.

`[INFERENCE]` These are errors on recordings the mixtures have not heard, so they are the relevant ones, but they come from one split.

## 7. Two ways of choosing a threshold

**What this section asks, and why it matters.** The article's rule is a one-vs-all ROC: species `k` against all other calls, the highest true-positive rate with a false-positive rate of at most 5 %. But the deployed
decision is not one-vs-all. A call is detected as `k` only if `k` **wins** the identification **and** `LLR_k` is above the threshold. A call that another model wins can have a high `LLR_k` and still never be detected as `k`, so letting it
constrain the threshold is inconsistent with the decision that is actually taken.

Both rules are computed for every species, and each is then evaluated with the **deployed decision**.

In [6]:
# Compute both threshold rules, show the false-positive rate a one-vs-all analysis assigns to each, and compare the deployed decisions.
rows = []
for k, code_ in enumerate(USABLE_CODES):
    positive, wins = true_index == k, hypothesis == k
    ova = D.one_vs_all_threshold(llr[:, k], positive, MAX_FPR)
    e2e = D.end_to_end_threshold(llr[:, k], wins, positive, MAX_FPR)
    negatives = ~positive
    rows.append({"code": code_, "positives": int(positive.sum()), "one_vs_all": ova, "end_to_end": e2e,
                 "one_vs_all_FPR_at_one_vs_all": float(((llr[:, k] >= ova) & negatives).sum() / negatives.sum()),
                 "one_vs_all_FPR_at_end_to_end": float(((llr[:, k] >= e2e) & negatives).sum() / negatives.sum()),
                 "non_winning_calls_above_end_to_end": int((negatives & ~wins & (llr[:, k] >= e2e)).sum())})
thresholds_table = pd.DataFrame(rows).set_index("code")
display(thresholds_table.round(4))

def deployed_metrics(thresholds: dict) -> pd.DataFrame:
    """Per-species metrics of the deployed decision for a given set of thresholds."""
    out = D.decide(scores, np.array([thresholds[c] for c in USABLE_CODES]))
    table = {}
    for k, code_ in enumerate(USABLE_CODES):
        table[code_] = D.confusion_counts(true_index == k, (out["hypothesis"] == k) & out["accepted"])
    return pd.DataFrame(table).T


ova_thresholds = thresholds_table.one_vs_all.to_dict()
e2e_thresholds = thresholds_table.end_to_end.to_dict()
metrics_ova, metrics_e2e = deployed_metrics(ova_thresholds), deployed_metrics(e2e_thresholds)
print("deployed decision with the one-vs-all thresholds:")
display(metrics_ova.round(4))
print("deployed decision with the end-to-end thresholds:")
display(metrics_e2e.round(4))

def macro(metrics, thresholds):
    out = D.decide(scores, np.array([thresholds[c] for c in USABLE_CODES]))
    return {"accepted_fraction": float(out["accepted"].mean()),
            "correct_and_accepted": float((out["accepted"] & (out["hypothesis"] == true_index)).mean()),
            "macro_recall": float(metrics.recall.mean()), "macro_precision": float(metrics.precision.mean()), "macro_FPR": float(metrics.FPR.mean())}

display(pd.DataFrame({"one-vs-all": macro(metrics_ova, ova_thresholds), "end-to-end": macro(metrics_e2e, e2e_thresholds)}).T.round(4))

,positives,one_vs_all,end_to_end,one_vs_all_FPR_at_one_vs_all,one_vs_all_FPR_at_end_to_end,non_winning_calls_above_end_to_end
code,,,,,,
f01,19,8.0751,8.0751,0.0000,0.0000,0
f02,24,8.5813,3.7012,0.0056,0.4022,141
f03,26,6.9382,6.9382,0.0028,0.0028,1
f04,47,1.9181,5.0647,0.0000,0.0000,0
f05,135,5.5535,5.5535,0.0081,0.0081,1
f06,13,5.2310,5.2310,0.0027,0.0027,1
f07,17,5.7501,5.7501,0.0027,0.0027,1
f08,25,7.1894,7.1894,0.0000,0.0000,0
f10,76,9.0146,9.0146,0.0065,0.0065,1


deployed decision with the one-vs-all thresholds:


,TP,FN,FP,TN,recall,precision,FPR
f01,19.0,0.0,0.0,363.0,1.0000,1.0000,0.0000
f02,3.0,21.0,2.0,356.0,0.1250,0.6000,0.0056
f03,26.0,0.0,0.0,356.0,1.0000,1.0000,0.0000
f04,46.0,1.0,0.0,335.0,0.9787,1.0000,0.0000
f05,129.0,6.0,1.0,246.0,0.9556,0.9923,0.0040
f06,13.0,0.0,0.0,369.0,1.0000,1.0000,0.0000
f07,17.0,0.0,0.0,365.0,1.0000,1.0000,0.0000
f08,25.0,0.0,0.0,357.0,1.0000,1.0000,0.0000
f10,76.0,0.0,1.0,305.0,1.0000,0.9870,0.0033


deployed decision with the end-to-end thresholds:


,TP,FN,FP,TN,recall,precision,FPR
f01,19.0,0.0,0.0,363.0,1.0000,1.0000,0.0000
f02,22.0,2.0,3.0,355.0,0.9167,0.8800,0.0084
f03,26.0,0.0,0.0,356.0,1.0000,1.0000,0.0000
f04,46.0,1.0,0.0,335.0,0.9787,1.0000,0.0000
f05,129.0,6.0,1.0,246.0,0.9556,0.9923,0.0040
f06,13.0,0.0,0.0,369.0,1.0000,1.0000,0.0000
f07,17.0,0.0,0.0,365.0,1.0000,1.0000,0.0000
f08,25.0,0.0,0.0,357.0,1.0000,1.0000,0.0000
f10,76.0,0.0,1.0,305.0,1.0000,0.9870,0.0033


,accepted_fraction,correct_and_accepted,macro_recall,macro_precision,macro_FPR
one-vs-all,0.9372,0.9267,0.8955,0.9533,0.0014
end-to-end,0.9895,0.9764,0.9834,0.9844,0.0017


`[OBSERVED]` The two rules give the **same threshold for seven of the nine species**. They differ for `f02` (8.5813 with the one-vs-all rule, **3.7012** end to end) and for `f04` (1.9181 and 5.0647).

`[OBSERVED]` The reason is visible for `f02`. At its end-to-end threshold a one-vs-all analysis would count **141 calls** that `f02` does not win, and a false-positive rate of **0.4022** against the ceiling of 0.05. Those calls have a high `LLR_f02`
but another model wins them, so they can never be detected as `f02`. With the one-vs-all rule they keep the threshold at 8.5813, where the deployed detector accepts only 3 of the 24 true `f02` calls (recall 0.1250). With the end-to-end rule it accepts 22 (recall **0.9167**) at the cost of one more false positive (3 instead of 2).

`[OBSERVED]` For `f04` the two thresholds give identical deployed results (46 detected, 1 missed, no false positive): the one-vs-all value is lower than needed, and the end-to-end rule takes the largest threshold among equals.

`[OBSERVED]` Over all species the accepted fraction goes from **0.9372 to 0.9895**, the share of calls that are both accepted and correct from 0.9267 to 0.9764, the macro recall from 0.8955 to 0.9834 and the macro precision from 0.9533 to 0.9844, with the macro false-positive rate moving from 0.0014 to 0.0017.

`[INFERENCE]` The one-vs-all rule is not wrong; it answers a question the detector does not ask. The rule that matches the deployed decision is the one used from here on.

## 8. How much margin do the thresholds have?

**What this section asks, and why it matters.** The end-to-end rule keeps the false-positive rate within 5 % and takes the highest recall, but the only negatives in calibration are the calls of the other frog species, and they almost
never win the species they are not. The false-positive constraint then hardly binds, and the highest recall is reached by accepting every correctly identified call. The threshold is therefore pushed down to the **lowest ratio among the true calls
that win**, with no margin, so one weak call decides it. This section measures that.

In [7]:
# Per species: the lowest accepted true ratio, the spread of the true ratios, and how many calibration calls and recordings stand behind each threshold.
margin_rows = []
for k, code_ in enumerate(USABLE_CODES):
    positive_wins = (true_index == k) & (hypothesis == k)
    own = llr[positive_wins, k]
    margin_rows.append({
        "code": code_, "threshold": e2e_thresholds[code_], "lowest_true_llr": own.min(), "q05_true_llr": np.quantile(own, 0.05),
        "median_true_llr": np.median(own), "threshold_is_lowest_true_llr": bool(np.isclose(e2e_thresholds[code_], own.min())),
        "calibration_calls": int((true_index == k).sum()), "calibration_recordings": int(cal[cal.code == code_].recording.nunique()),
        "false_positives": int(metrics_e2e.loc[code_, "FP"]),
    })
margins = pd.DataFrame(margin_rows).set_index("code")
display(margins.round(3))
print("species whose threshold is exactly their lowest true ratio:", int(margins.threshold_is_lowest_true_llr.sum()), "of", len(margins))

,threshold,lowest_true_llr,q05_true_llr,median_true_llr,threshold_is_lowest_true_llr,calibration_calls,calibration_recordings,false_positives
code,,,,,,,,
f01,8.075,8.075,8.683,10.722,True,19,1,0
f02,3.701,3.701,3.779,5.891,True,24,2,3
f03,6.938,6.938,7.133,9.023,True,26,1,0
f04,5.065,5.065,6.257,10.562,True,47,3,0
f05,5.554,5.554,7.295,10.797,True,135,1,1
f06,5.231,5.231,5.391,8.475,True,13,2,0
f07,5.750,5.750,6.677,7.469,True,17,1,0
f08,7.189,7.189,7.976,9.328,True,25,2,0
f10,9.015,9.015,9.912,12.965,True,76,5,1


species whose threshold is exactly their lowest true ratio: 9 of 9


`[OBSERVED]` For **all 9 species** the threshold equals the lowest ratio among the true calls that win: the rule accepts every correctly identified calibration call, and nothing more. The margin above the lowest true ratio is small: the 5th percentile of the true ratios is 3.779 for `f02` (threshold 3.701), 5.391
for `f06` (5.231) and 7.133 for `f03` (6.938).

`[OBSERVED]` The false-positive ceiling hardly constrains anything: the most false positives for a species is 3 (`f02`, a rate of 0.0084), against the 5 % allowed.

`[INFERENCE]` With only frog calls as negatives, and those calls rarely winning another species, the ceiling is never reached, so the highest recall is simply reached by accepting every correctly identified call. Each threshold is then the **weakest true call of the calibration set**: a true call in the
field weaker than it is rejected, and one call in one or a few recordings decides the value. The calibration set also contains no environmental noise as negatives, so the rule says nothing about how many noise segments the thresholds let through; the article addressed that by fine-tuning on annotated field samples, which were not delivered.

## 9. Do the thresholds depend on the split?

**What this section asks, and why it matters.** Everything above comes from one split of 18 calibration recordings, and some species have two or three recordings in calibration. If a different split gave
very different thresholds, the numbers above would describe that split and not the detector. The whole procedure (search a split, fit on TRAIN, score, choose thresholds) is repeated for 20 different split seeds, the first being the one used here.

In [8]:
# Repeat the split, fit, score and threshold procedure for 20 seeds, and summarise each species' threshold and calibration recall.
N_SPLITS = 20


def one_split(seed):
    """Thresholds and calibration recall obtained from the split of one seed."""
    sp = D.calibration_split(corpus.calls, USABLE_CODES, CALIBRATION_FRACTION, seed, SPLIT_ATTEMPTS)
    side_train = usable[usable.recording.isin(sp["train"])]
    side_cal = usable[usable.recording.isin(sp["calibration"])].reset_index(drop=True)
    models = {c: fit_gmm(corpus.frames(tuple(side_train[side_train.code == c].call_id)), N_COMPONENTS,
                         MODEL_SEED_BASE + species_number(c)) for c in USABLE_CODES}
    s = score_calls(models, corpus, list(side_cal.call_id))
    l, h = D.llr_matrix(s), s.argmax(axis=1)
    t = np.array([USABLE_CODES.index(c) for c in side_cal.code])
    rows = []
    for k, c in enumerate(USABLE_CODES):
        thr = D.end_to_end_threshold(l[:, k], h == k, t == k, MAX_FPR)
        m = D.confusion_counts(t == k, (h == k) & (l[:, k] >= thr))
        rows.append({"seed": seed, "code": c, "threshold": thr, "recall": m["recall"], "FPR": m["FPR"],
                     "calibration_calls": int((t == k).sum()), "calibration_recordings": int(side_cal[side_cal.code == c].recording.nunique())})
    return rows


stability = pd.DataFrame([r for rows in Parallel(n_jobs=-1)(delayed(one_split)(SPLIT_SEED + s) for s in range(N_SPLITS)) for r in rows])
assert stability[stability.seed == SPLIT_SEED].set_index("code").threshold.round(9).equals(thresholds_table.end_to_end.round(9)), \
    "the first seed must reproduce the thresholds above"

stability_summary = stability.groupby("code").agg(
    threshold_mean=("threshold", "mean"), threshold_std=("threshold", "std"), threshold_min=("threshold", "min"),
    threshold_max=("threshold", "max"), recall_mean=("recall", "mean"), recall_min=("recall", "min"),
    FPR_max=("FPR", "max"), calibration_recordings_mean=("calibration_recordings", "mean"))
display(stability_summary.round(3))
print("splits:", N_SPLITS, "| the first seed reproduces the thresholds above: True")

,threshold_mean,threshold_std,threshold_min,threshold_max,recall_mean,recall_min,FPR_max,calibration_recordings_mean
code,,,,,,,,
f01,8.019,1.055,6.568,10.120,1.000,1.000,0.000,1.00
f02,4.628,0.793,3.244,6.103,0.972,0.917,0.023,2.15
f03,6.576,1.087,4.451,8.112,0.973,0.794,0.000,2.05
f04,4.772,1.608,2.668,8.651,0.990,0.963,0.000,2.15
f05,6.079,0.524,5.444,7.157,0.976,0.941,0.004,1.00
f06,4.256,2.152,2.276,10.222,1.000,1.000,0.017,1.05
f07,3.378,1.067,1.326,5.750,0.921,0.700,0.002,1.80
f08,6.676,1.417,4.409,8.405,0.998,0.968,0.000,1.75
f10,10.393,2.647,5.435,15.404,1.000,1.000,0.009,3.10


splits: 20 | the first seed reproduces the thresholds above: True


`[OBSERVED]` Across the 20 split seeds the thresholds move a lot: the standard deviation is between 0.524 (`f05`) and **2.647** (`f10`) ratio units, and the range of a single species is wide: **`f10` 5.435 to 15.404**, `f06` 2.276 to 10.222, `f04` 2.668 to 8.651, `f01` 6.568 to 10.120.
The calibration side has on average only 1.00 recordings for `f01` and `f05` and 1.05 for `f06`.

`[OBSERVED]` The calibration recall also varies: its lowest value is **0.700** for `f07` and 0.794 for `f03`, and `f07`'s mean recall is 0.921. The highest false-positive rate in any split is 0.023 (`f02`).

`[OBSERVED]` The frozen `f07` threshold, 5.7501, is the **largest** of the 20 values for that species (mean 3.378).

`[INFERENCE]` The thresholds describe which recordings fell into calibration at least as much as they describe the species. The frozen set is one draw from that spread, and a threshold that is high for its species (such as `f07`'s) rejects more true calls than a typical split would.

## 10. The article's thresholds, for provenance

**What this section asks, and why it matters.** The article's threshold vector is shown next to this notebook's, with no suggestion that they should agree: they belong to different features, mixture sizes and
score distributions, and the article tuned its vector on annotated field samples that were not delivered. `f09` has no model here.

In [9]:
# Show the article's final threshold vector next to the thresholds selected here.
PAPER_THRESHOLDS = {"f01": 5.0, "f02": 3.0, "f03": 6.0, "f04": 9.0, "f05": 6.75, "f06": 5.25, "f07": 5.5, "f08": 11.0, "f09": 6.0, "f10": 6.0}
comparison = pd.DataFrame({"paper": PAPER_THRESHOLDS, "this_notebook": e2e_thresholds}).round(4)
comparison["used_by_detector"] = comparison.index.isin(USABLE_CODES)
display(comparison)

,paper,this_notebook,used_by_detector
f01,5.00,8.0751,True
f02,3.00,3.7012,True
f03,6.00,6.9382,True
f04,9.00,5.0647,True
f05,6.75,5.5535,True
f06,5.25,5.2310,True
f07,5.50,5.7501,True
f08,11.00,7.1894,True
f09,6.00,NaN,False
f10,6.00,9.0146,True


`[OBSERVED]` The article's thresholds range from 3 to 11 and these from 3.7012 to 9.0146, with no consistent relation: `f04` is 9.00 in the article and 5.0647 here, `f08` 11.00 and 7.1894, `f10` 6.00 and 9.0146.

`[INFERENCE]` As expected, they are not comparable: the features, mixture size and score distributions differ, and the article tuned its vector on annotated field samples.

## 11. Freeze the detector

**What this section asks, and why it matters.** The models and thresholds are written together, with the hash of every model file, so that nothing can change a model after its thresholds were chosen without being noticed. The detector
already on disk is compared first: when this notebook reproduces it, it is **kept**, so that everything computed from it stays exactly valid.

In [10]:
# Compare with the detector on disk, keep its model files if identical, otherwise write the new ones, and reload and check it.
manifest = {
    "detector_status": "frozen",
    "notebook": "06_detector_calibration",
    "feature": FEATURE_NAME, "rasta_start": selection["rasta_start"], "feature_dimensions": 20, "n_components": N_COMPONENTS,
    "excluded_class": selection["excluded_class"],
    "threshold_rule": "accept winning hypothesis k iff LLR_k >= threshold_k",
    "threshold_selection": f"end-to-end: highest recall with FPR <= {MAX_FPR}, largest threshold among ties",
    "llr_definition": "class mean log-likelihood minus the median of the alternative models' mean log-likelihoods",
    "validation_unit": "source_wav",
    "calibration_split": {"seed": SPLIT_SEED, "attempts": SPLIT_ATTEMPTS, "selected_attempt": split["attempt"],
                          "fraction_target": CALIBRATION_FRACTION, "train_recordings": len(TRAIN_RECORDINGS),
                          "calibration_recordings": len(CALIBRATION_RECORDINGS)},
    "model_seed_base": MODEL_SEED_BASE,
}

identical = False
if (MODELS_DIR / "final_detector_manifest.json").exists():
    try:
        on_disk_models, on_disk_thresholds, _ = D.load_detector(MODELS_DIR)
        identical = (list(on_disk_models) == list(USABLE_CODES)
                     and all(D.parameters_equal(on_disk_models[c], FINAL_MODELS[c]) for c in USABLE_CODES)
                     and all(abs(on_disk_thresholds[c] - e2e_thresholds[c]) < 1e-9 for c in USABLE_CODES))
    except (ValueError, KeyError, FileNotFoundError):
        identical = False
print("detector on disk is identical to this one:", identical)

D.save_detector(MODELS_DIR, FINAL_MODELS, e2e_thresholds, manifest, rewrite_models=not identical)
print("model files kept as they were; thresholds and manifest rewritten" if identical else "detector written")

reloaded_models, reloaded_thresholds, reloaded_manifest = D.load_detector(MODELS_DIR)
assert all(D.parameters_equal(reloaded_models[c], FINAL_MODELS[c]) for c in USABLE_CODES)
assert all(abs(reloaded_thresholds[c] - e2e_thresholds[c]) < 1e-9 for c in USABLE_CODES)
print("reloaded detector matches the models and thresholds above: True")
print("model files verified against their hashes:", len(reloaded_manifest["gmm_sha256"]))
print(json.dumps({c: round(v, 4) for c, v in reloaded_thresholds.items()}))

detector on disk is identical to this one: True
model files kept as they were; thresholds and manifest rewritten
reloaded detector matches the models and thresholds above: True
model files verified against their hashes: 9
{"f01": 8.0751, "f02": 3.7012, "f03": 6.9382, "f04": 5.0647, "f05": 5.5535, "f06": 5.231, "f07": 5.7501, "f08": 7.1894, "f10": 9.0146}


`[OBSERVED]` The detector on disk is **identical** to the one built here: the same models, the same thresholds. The model files were kept as they were, so their hashes are unchanged, and the thresholds and manifest were rewritten. The reloaded detector matches
the models and thresholds above, and all 9 model files agree with the hashes in the manifest.

## 12. Save the calibration tables

**What this section asks, and why it matters.** The tables behind the numbers above are written so that they can be inspected without refitting. The split is stored with its provenance, so that the calibration recordings can be identified later.

In [11]:
# Write the calibration tables and the split, and list only the files this notebook wrote.
scores_table = pd.concat([cal[["call_id", "code", "wav"]].reset_index(drop=True),
                          pd.DataFrame(scores, columns=[f"score_{c}" for c in USABLE_CODES]),
                          pd.DataFrame(llr, columns=[f"llr_{c}" for c in USABLE_CODES])], axis=1)
scores_table["hypothesis"] = [USABLE_CODES[i] for i in hypothesis]

OUTPUTS = {
    "calibration_scores_and_llr.csv": scores_table,
    "thresholds_by_rule.csv": thresholds_table.reset_index(),
    "metrics_one_vs_all.csv": metrics_ova.reset_index(names="code"),
    "metrics_end_to_end.csv": metrics_e2e.reset_index(names="code"),
    "threshold_margins.csv": margins.reset_index(),
    "threshold_stability_by_split.csv": stability,
    "threshold_stability_summary.csv": stability_summary.reset_index(),
    "model_audit.csv": pd.DataFrame(model_rows),
}
for name, table in OUTPUTS.items():
    table.to_csv(RESULTS_DIR / name, index=False)
(RESULTS_DIR / "train_calibration_recordings.json").write_text(json.dumps({
    "train_recordings": sorted(TRAIN_RECORDINGS), "calibration_recordings": sorted(CALIBRATION_RECORDINGS),
    "split_seed": SPLIT_SEED, "split_search_attempts": SPLIT_ATTEMPTS, "selected_attempt": split["attempt"],
    "calibration_fraction_target": CALIBRATION_FRACTION}, indent=2), encoding="utf-8")

print("written to", RESULTS_DIR.relative_to(REPO_ROOT).as_posix())
for name in [*OUTPUTS, "train_calibration_recordings.json"]:
    print(f"  {name:<40} {(RESULTS_DIR / name).stat().st_size / 1e6:>7.3f} MB")

written to data/processed/final_detector_calibration
  calibration_scores_and_llr.csv             0.150 MB
  thresholds_by_rule.csv                     0.001 MB
  metrics_one_vs_all.csv                     0.000 MB
  metrics_end_to_end.csv                     0.000 MB
  threshold_margins.csv                      0.001 MB
  threshold_stability_by_split.csv           0.009 MB
  threshold_stability_summary.csv            0.001 MB
  model_audit.csv                            0.001 MB
  train_calibration_recordings.json          0.003 MB


## 13. Conclusions and limitations

**What this notebook delivers.** The detector of the article, calibrated on recordings the models have not heard: one frozen mixture per species (PLP-RASTA, 4 Gaussians), the likelihood-ratio verification, and one threshold per species chosen with the rule that matches the deployed decision. The models and thresholds are in
`models/gmm_robust_final`, with a manifest of model hashes. The code (split search, ratios, threshold rules, saving and loading) is in `frog_presence.detector` with tests, and reproduces the detector that was already frozen exactly: the same
calibration recordings, the same model parameters and the same thresholds.

**What was found.**

1. **Identification is 97.644 % correct on unseen recordings** (9 errors in 382 calls), with `f05`, `f02` and `f04` the confused ones.
2. **The article's one-vs-all rule cannot be applied to the deployed decision.** For `f02` it keeps the threshold at 8.5813 and accepts 3 of 24 true calls; matching the rule to the decision gives 3.7012 and 22 of 24, and lifts the macro recall from 0.8955 to 0.9834.
3. **Every threshold is the weakest true call of the calibration set** (9 of 9), with no margin.
4. **The thresholds depend on the split.** Their standard deviation over 20 splits is up to 2.647 ratio units and `f10` ranges from 5.435 to 15.404.

**Decisions taken here, which the article does not state.**

1. **The split is recording-level and global**, with about 25 % of each species in calibration, and the closest of 10000 random candidates is kept.
2. **The mixtures are fitted on all TRAIN calls**, not on the training duration of the selection experiment, and are not refitted after calibration.
3. **The median of the alternatives is taken over mean log-likelihoods.**
4. **The end-to-end rule**: the highest recall with a false-positive rate of at most 5 %, and the largest threshold among ties.
5. **Existing models are kept when reproduced**, so that what was computed from them stays valid.

**Limitations.**

- **The calibration figures are not an independent test.** The thresholds are chosen on the calibration calls and reported on them, and the calibration recordings were also part of the corpus used in notebook 05 to choose the model order.
- **One split, and few recordings behind it.** Four species have a single calibration recording, and the thresholds change a great deal with the split (section 9).
- **No environmental negatives.** The negatives are the calls of the other frog species, so the false-positive rate says nothing about noise, rain, insects or unmodelled species, which dominate the field recordings.
- **The article's field fine-tuning cannot be reproduced**, because the annotated pond samples were not delivered.
- **The final models do not use the calibration recordings** (18 of 66, about a quarter of the labelled material).
- **`f09` has no model.**
- **The PLP-RASTA start is the zero start**, with the dependence on the start-up transient found in notebook 04.

**What the next stages inherit.** A detector whose thresholds sit at the weakest calibration call, calibrated against frog calls only. Applied to long recordings, where almost every candidate segment is noise, it has nothing in its calibration that controls false alarms, so the detections it accepts in the field must be read
with that in mind, and the sensitivity of the field results to the thresholds has to be examined, not assumed.